# W6D5 — CLIP: Searching Pictures by Typing a Sentence — Lab

**Week 6 · Day 5 · Representation Learning** · Lab

All week you produced vectors and never said what they were for. Today they get a purpose, and it
is the one most shipped systems actually want: **search**.

CLIP puts images and text in one space. That single fact means a cosine between a sentence and a
photograph is a meaningful number, and everything in this lab is a consequence: text-to-image
search, image-to-text search, and classification with no training images at all — the class name is
the classifier.

You start from the two images and three captions on this morning's slide, hard-coded as 2-dim
vectors, and reproduce the six numbers including **0.991**, **0.348** and the vague caption's
**0.781** against both images. Then 200 real pairs, retrieval metrics in both directions, a
zero-shot number that goes into the same row as Thursday's probe — and the graded task, which is
finding where it breaks.

**Time budget:** ~115 minutes. Sections 1–2 are the lab; Section 3 is a stretch you may finish at home.

<div dir="rtl" align="right">

# الأسبوع ٦ · اليوم ٥ — CLIP: البحث في الصور بكتابة جملة

**الأسبوع السادس · اليوم الخامس · تعلّم التمثيل** · معمل

أنتجتَ المتّجهات طوال الأسبوع ولم تقل لماذا. واليوم تنال غايةً، وهي الغاية التي تريدها معظم الأنظمة
المُشغَّلة فعلًا: **البحث**.

يضع CLIP الصور والنصّ في فضاء واحد. وهذه الحقيقة وحدها تعني أن جيب التمام بين جملة وصورة عدد ذو
معنى، وكل ما في هذا المعمل نتيجةٌ لها: البحث من النص إلى الصورة، ومن الصورة إلى النص، والتصنيف بلا
صور تدريب البتّة — فاسم الفئة هو المصنّف.

تبدأ من الصورتين والتعليقات الثلاثة على شريحة هذا الصباح، مكتوبةً بثبوت متّجهاتٍ ثنائية البُعد،
وتُعيد إنتاج الأعداد الستّة ومنها **٠٫٩٩١** و**٠٫٣٤٨** ودرجة التعليق الغامض **٠٫٧٨١** مقابل
الصورتين كلتيهما. ثم مئتا زوج حقيقي، ومقاييس استرجاع في الاتجاهين، ورقم بلا تدريب يدخل الصفّ نفسه مع
فحص الخميس — والمهمّة المُقيَّمة، وهي أن تجد أين ينكسر.

**الزمن المتوقّع:** نحو ١١٥ دقيقة. القسمان الأول والثاني هما المعمل، والقسم الثالث إضافي يمكن إكماله في المنزل.

</div>

> **This is your lab notebook.** Work through the hints — they tell you what to do and where
> to look, not what to type. Stuck for more than ten minutes on one task? Open the `_guided`
> version. That is not cheating; sitting stuck in silence is the only mistake. The full
> solution is released at the end of the day.

<div dir="rtl" align="right">

> **هذا دفتر المعمل الخاص بك.** اعمل وفق الإرشادات — فهي تخبرك بما يجب فعله وأين تبحث، لا بما
> تكتبه حرفيًا. إذا توقّفت أكثر من عشر دقائق عند مهمة واحدة فافتح نسخة `_guided`؛ هذا ليس غشًّا،
> والخطأ الوحيد هو أن تبقى متوقّفًا بصمت. ويُنشر الحل الكامل في نهاية اليوم.

</div>

## Learning objectives

By the end of this lab you can:

- Build a full cosine similarity matrix between two sets of vectors and read it in both directions.
- Say what "a shared embedding space" buys you, and check that a claimed one really is shared.
- Compute Recall@1, Recall@5 and mean reciprocal rank, and say what each one hides.
- Classify with no training images at all, and say what that number does and does not establish.
- Measure how much a zero-shot result depends on the wording of the prompt.
- Find a retrieval failure, name its cause, and say which failures are properties of the training
  data rather than bugs.
- Estimate the cost of brute-force search at scale, and say when an index becomes necessary.

<div dir="rtl" align="right">

## أهداف التعلّم

في نهاية هذا المعمل تستطيع:

- أن تبني مصفوفة تشابه جيب تمام كاملة بين مجموعتَي متّجهات وأن تقرأها في الاتجاهين.
- أن تقول ماذا يشتري لك «فضاء تضمين مشترك»، وأن تفحص أن المُدَّعى مشتركٌ فعلًا.
- أن تحسب الاستدعاء عند ١ وعند ٥ ومتوسّط الرتبة المتبادلة، وأن تقول ماذا يُخفي كلٌّ منها.
- أن تُصنّف بلا صور تدريب إطلاقًا، وأن تقول ماذا يُثبت ذلك الرقم وماذا لا يُثبت.
- أن تقيس مقدار اعتماد نتيجةٍ بلا تدريب على صياغة الموجّه.
- أن تجد إخفاق استرجاع، وتُسمّي سببه، وتقول أيّ الإخفاقات خصائص لبيانات التدريب لا أخطاء برمجية.
- أن تُقدّر كلفة البحث الشامل على نطاق واسع، وأن تقول متى يصير الفهرس ضرورة.

</div>

## About the data

**`image_captions_small`** — 200 image–caption pairs, new this week. The images are whole scenes
from the same CC-BY COCO pool as `small_image_5class`, with **no image id in common** with it, at a
longest side of 384. Each image has exactly one caption known to be its own, which is what makes
Recall@1 and mean reciprocal rank computable at all: without a known-correct answer per query,
retrieval has no ground truth and you can only look at the results and nod.

192 of the captions are COCO's own crowd-written sentences. **The last eight were written for this
course, and they are there to fail**: three counting captions, two spatial relations, one caption
about text inside the picture, and one Arabic caption paired with an English twin on a different
image. `captions.csv` names the kind in a `hard_case` column. Every one of the eight is *true of
its image* — the counts and the left/right relations were read off COCO's own bounding boxes — so a
wrong retrieval is the model's failure and not a mislabelled row.

**`small_image_5class`** — the week's 400 images again, for the zero-shot classification comparison
against Thursday's probe. And `dino_features.parquet` from Thursday, so the comparison is on the
same images.

**The known problem, stated up front:** the Arabic caption performs far worse than its English
twin. That is the finding, not a broken file. CLIP was trained on English-dominated web alt text,
and the gap is the lab making that visible rather than describing it.

**First run downloads** `openai/clip-vit-base-patch32` — 605 MB. Embedding 200 images and 200
captions takes about **3 seconds**; the whole notebook is under a minute of compute.

<div dir="rtl" align="right">

## عن البيانات

**`image_captions_small`** — مئتا زوج من صورة وتعليق، جديدة هذا الأسبوع. والصور مشاهد كاملة من
مجموعة COCO المرخّصة نفسها التي جاءت منها `small_image_5class`، و**لا معرّف صورة مشترك** بينهما،
وأطول ضلع فيها ٣٨٤. ولكل صورة تعليق واحد معروف أنه لها، وهذا ما يجعل الاستدعاء عند ١ ومتوسّط الرتبة
المتبادلة قابلين للحساب أصلًا: فبلا جوابٍ صحيح معروف لكل استعلام لا حقيقة أرضية للاسترجاع ولا يسعك
إلا النظر إلى النتائج والإيماء.

و١٩٢ من التعليقات جمل COCO التي كتبها بشر. **أما الثمانية الأخيرة فكُتبت لهذه الدورة، وهي هناك
لتفشل**: ثلاثة تعليقات عدّ، وعلاقتان مكانيتان، وتعليق عن نصّ داخل الصورة، وتعليق عربي مع توأم
إنجليزي على صورة أخرى. ويُسمّي `captions.csv` النوع في عمود `hard_case`. وكل واحد من الثمانية **صحيح
في صورته** — فالأعداد وعلاقات اليمين واليسار قُرئت من صناديق COCO نفسها — فالاسترجاع الخاطئ إخفاق
النموذج لا صفٌّ مُساء تسميته.

**`small_image_5class`** — صور الأسبوع الأربعمئة مرّةً أخرى، لمقارنة التصنيف بلا تدريب بفحص الخميس.
وملف `dino_features.parquet` من الخميس لتكون المقارنة على الصور نفسها.

**والمشكلة المعروفة، مذكورةً سلفًا:** أداء التعليق العربي أسوأ بكثير من توأمه الإنجليزي. وهذه هي
النتيجة لا خللٌ في ملف. فقد دُرِّب CLIP على نصوص بديلة من الشبكة تغلب عليها الإنجليزية، والفجوة هي
المعمل يُظهر ذلك بدل أن يصفه.

**التشغيل الأول ينزّل** النموذج `openai/clip-vit-base-patch32` — ‏٦٠٥ ميجابايت. وتضمين مئتَي صورة
ومئتَي تعليق نحو **ثلاث ثوانٍ**؛ وحساب الدفتر كله دون دقيقة.

</div>

## Setup

One API note that matters and has changed between `transformers` versions: `get_image_features` and
`get_text_features` return an **output object**, and the vector is on `.pooler_output`. It is not
normalised. The helpers in the setup cell do the normalisation explicitly, because on this course
you should be able to point at the line where a vector became unit-length.

<div dir="rtl" align="right">

## الإعداد

ملاحظة واحدة عن الواجهة تهمّ وقد تغيّرت بين إصدارات `transformers`: تُعيد `get_image_features` و
`get_text_features` **كائن خرج**، والمتّجه في `.pooler_output`. وهو غير مُطبَّع. وتقوم الدوال
المساعدة في خلية الإعداد بالتطبيع صراحةً، لأنك في هذه الدورة ينبغي أن تستطيع الإشارة إلى السطر الذي
صار فيه المتّجه بطول الوحدة.

</div>

In [ ]:
# === AIEP portable setup — works locally (conda) and on Google Colab ===============
try:
    import aiep
except ImportError:
    import subprocess, sys
    from pathlib import Path
    _local = next((p / "shared" for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "shared" / "aiep").is_dir()), None)
    if _local:
        sys.path.insert(0, str(_local))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "git+https://github.com/0xRush/AIEP_Olo_student.git#subdirectory=shared"])
    import aiep

from aiep.env import ensure, seed_everything, device, versions
from aiep.data import get_dataset_dir, load_artefact
from aiep.paths import ARTEFACT_DIR
from aiep.checks import check, check_close, check_shape, report
from aiep.viz import use_course_style, savefig

ensure("transformers", "torch", "matplotlib", "pandas", "pyarrow")
seed_everything(42)

import textwrap
import time

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image

use_course_style()
np.set_printoptions(precision=3, suppress=True)
torch.set_grad_enabled(False)          # nothing is trained today, and nothing needs to be

SEED = 42
CHECKPOINT = "openai/clip-vit-base-patch32"
IMAGE_BATCH = 32

CAPTION_ROOT = get_dataset_dir("image_captions_small")
CAPTIONS = pd.read_csv(CAPTION_ROOT / "captions.csv")
CAPTION_IMAGES = [CAPTION_ROOT / "images" / name for name in CAPTIONS.file]

CLASS_ROOT = get_dataset_dir("small_image_5class") / "images"
CLASS_PATHS = sorted(CLASS_ROOT.rglob("*.jpg"))
CLASS_LABELS = np.array([p.parent.name for p in CLASS_PATHS])
CLASSES = sorted(str(name) for name in set(CLASS_LABELS))

# Thursday's numbers, for the three-way comparison in task 2.4.
DINO_FRAME = pd.read_parquet(load_artefact("dino_features.parquet"))
PROBE_RESULTS = pd.read_parquet(load_artefact("probe_results.parquet"))
D4_PROBE = float(PROBE_RESULTS.loc[(PROBE_RESULTS.representation == "DINOv2 features")
                                   & (PROBE_RESULTS.evaluation == "linear probe"),
                                   "accuracy"].iloc[0])
W4D5_FINETUNE = float(PROBE_RESULTS.loc[PROBE_RESULTS.representation.str.contains("fine-tuned"),
                                        "accuracy"].iloc[0])

print(f"{len(CAPTIONS)} caption pairs, {CAPTIONS.hard_case.notna().sum()} of them written to fail")
print(CAPTIONS.hard_case.value_counts().to_string())
print(f"\n{len(CLASS_PATHS)} class images, classes {CLASSES}")
print(f"Thursday's DINOv2 probe {D4_PROBE:.4f} | W4D5's fine-tune {W4D5_FINETUNE:.4f}")
print(versions(), "| device:", device())

## Section 1 — Warm-up: six numbers by hand  (≈25 min)

Everything here works. Two images and three captions, as the slide's 2-dim vectors:

```
i₁ = [0.9, 0.1]   a cat        t₁ = [0.8, 0.2]   "a cat"
i₂ = [0.1, 0.9]   a car        t₂ = [0.2, 0.8]   "a car"
                               t₃ = [0.5, 0.5]   "a thing"
```

Build the full 2×3 cosine matrix and print all six numbers. Row 1 reads **0.991, 0.348, 0.781**:
the cat image against "a cat", against "a car", and against "a thing".

Three readings come out of that one table, and they are the whole of CLIP:

1. **Retrieval.** The argmax of a row is the best caption for that image.
2. **Zero-shot classification.** If the captions are class names, that argmax *is* a prediction, and
   no classifier was trained.
3. **The vague caption.** `t₃` scores **0.781 against both images, identically.** It is not wrong —
   both pictures are things — and it is useless, because a caption that describes everything ranks
   above the true caption for anything it is not about. That is the failure mode of every embedding
   search you will ever build, in three numbers.

Change `t₃` and re-run. A caption has to be *specific* to be findable, not merely accurate.

<div dir="rtl" align="right">

## القسم الأول — الإحماء: ستّة أعداد باليد (نحو ٢٥ دقيقة)

كل ما هنا يعمل. صورتان وثلاثة تعليقات، بمتّجهات الشريحة ثنائية البُعد:

```
i₁ = [0.9, 0.1]   قطّة        t₁ = [0.8, 0.2]   «قطّة»
i₂ = [0.1, 0.9]   سيارة       t₂ = [0.2, 0.8]   «سيارة»
                              t₃ = [0.5, 0.5]   «شيء»
```

ابنِ مصفوفة جيب التمام ٢×٣ كاملةً واطبع الأعداد الستّة. ويُقرأ الصف الأول **٠٫٩٩١ و٠٫٣٤٨ و٠٫٧٨١**:
صورة القطّة مقابل «قطّة» ومقابل «سيارة» ومقابل «شيء».

وثلاث قراءات تخرج من هذا الجدول الواحد، وهي CLIP كله:

١. **الاسترجاع.** أكبر قيمة في الصف هي أفضل تعليق لتلك الصورة.
٢. **التصنيف بلا تدريب.** فإن كانت التعليقات أسماء فئات كانت تلك القيمة الكبرى **تنبّؤًا**، ولم
   يُدرَّب مصنّف.
٣. **التعليق الغامض.** يسجّل `t₃` **٠٫٧٨١ مقابل الصورتين، متطابقًا.** وليس خاطئًا — فكلتاهما شيء —
   وهو عديم النفع، لأن تعليقًا يصف كل شيء يعلو على التعليق الصحيح لأي شيء لا يخصّه. وهذا إخفاق كل
   بحث تضميني ستبنيه، في ثلاثة أعداد.

غيّر `t₃` وأعِد التشغيل. فالتعليق يجب أن يكون **محدَّدًا** ليكون قابلًا للإيجاد، لا صحيحًا فحسب.

</div>

In [ ]:
IMAGE_VECTORS = np.array([[0.9, 0.1],       # i1 — a cat
                          [0.1, 0.9]])      # i2 — a car
TEXT_VECTORS = np.array([[0.8, 0.2],        # t1 — "a cat"
                         [0.2, 0.8],        # t2 — "a car"
                         [0.5, 0.5]])       # t3 — "a thing"
IMAGE_NAMES = ["i1 — a cat", "i2 — a car"]
TEXT_NAMES = ['t1 "a cat"', 't2 "a car"', 't3 "a thing"']


def cosine_matrix(left, right):
    """Every pairwise cosine between two sets of row vectors."""
    left = left / np.linalg.norm(left, axis=1, keepdims=True)
    right = right / np.linalg.norm(right, axis=1, keepdims=True)
    return np.dot(left, right.T)


WARMUP_MATRIX = cosine_matrix(IMAGE_VECTORS, TEXT_VECTORS)

print(pd.DataFrame(WARMUP_MATRIX.round(3), index=IMAGE_NAMES, columns=TEXT_NAMES).to_string())
print(f"\nrow 1 reads {WARMUP_MATRIX[0].round(3).tolist()} — the slide's 0.991, 0.348, 0.781")

for row, name in enumerate(IMAGE_NAMES):
    print(f"argmax for {name}: {TEXT_NAMES[WARMUP_MATRIX[row].argmax()]}   "
          f"<- zero-shot classification, from the same table")

VAGUE_IDENTICAL = bool(np.isclose(WARMUP_MATRIX[0, 2], WARMUP_MATRIX[1, 2]))
print(f"\n't3 \"a thing\"' scores {WARMUP_MATRIX[0, 2]:.3f} against the cat and "
      f"{WARMUP_MATRIX[1, 2]:.3f} against the car — identical: {VAGUE_IDENTICAL}")
print("It is accurate about both and useful for neither. Specific beats correct.")

## Section 2 — Core: six tasks  (≈60 min)

1. Load CLIP, embed 200 images and 200 captions, and check the two spaces really are one.
2. Search both ways: text → image and image → text, from the same matrix.
3. Recall@1, Recall@5 and MRR, in both directions.
4. Zero-shot classification, and the three-way comparison with Thursday.
5. Prompt sensitivity, measured.
6. Find the failures and diagnose them. **This is the graded task.**

<div dir="rtl" align="right">

## القسم الثاني — الأساسي: ست مهام (نحو ٦٠ دقيقة)

١. حمّل CLIP، وضمّن مئتَي صورة ومئتَي تعليق، وتحقّق أن الفضاءين واحد فعلًا.
٢. ابحث في الاتجاهين: من النص إلى الصورة ومن الصورة إلى النص، من المصفوفة نفسها.
٣. الاستدعاء عند ١ وعند ٥ ومتوسّط الرتبة المتبادلة، في الاتجاهين.
٤. التصنيف بلا تدريب، والمقارنة الثلاثية مع الخميس.
٥. حساسية الموجّه، مقيسةً.
٦. جِد الإخفاقات وشخّصها. **وهذه هي المهمّة المُقيَّمة.**

</div>

### Task 2.1 — one space, two modalities

Load `CLIPModel` and its processor, embed all 200 images and all 200 captions, and normalise both.

Then check the premise. **Assert that the image embeddings and the text embeddings have the same
dimension** — if they did not, no cosine between them would be defined and the entire idea would be
incoherent. It is a one-line check for the most important architectural fact of the day, and it is
worth writing because "shared embedding space" is said so often that people stop hearing it as a
claim about shapes.

Normalise both sets to unit length. After that a dot product *is* the cosine, and every similarity
is guaranteed to lie in `[−1, 1]` — which the sanity check at the bottom verifies, because a
similarity of 1.4 means you forgot.

<div dir="rtl" align="right">

### المهمة ٢٫١ — فضاء واحد ووسيطان

حمّل `CLIPModel` ومعالجه، وضمّن الصور المئتين والتعليقات المئتين، وطبّع الاثنين.

ثم افحص المُقدَّمة. **افحص أن لتضمينات الصور وتضمينات النصّ البُعد نفسه** — فلولا ذلك لما كان جيب
التمام بينهما مُعرَّفًا ولكانت الفكرة كلها غير متماسكة. وهو فحص في سطر لأهمّ حقيقة معمارية في اليوم،
ويستحقّ الكتابة لأن «فضاء التضمين المشترك» تُقال كثيرًا حتى يكفّ الناس عن سماعها ادّعاءً عن الأشكال.

طبّع المجموعتين إلى طول الوحدة. فبعدها يكون الجداء القياسي **هو** جيب التمام، ويُضمن وقوع كل تشابه في
المجال `[−1, 1]` — وهو ما يتحقّق منه فحص السلامة في الأسفل، لأن تشابهًا بقيمة ١٫٤ يعني أنك نسيت.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) from transformers import CLIPModel, CLIPProcessor — load both from CHECKPOINT, call
#    .eval(), and work in batches of IMAGE_BATCH so memory stays flat.
# 2) get_image_features and get_text_features return an output object in current
#    transformers: the vector is on .pooler_output, and it is not normalised.
# 3) The processor needs padding=True and truncation=True for a batch of captions of
#    different lengths.
# Search: "transformers CLIPModel get_image_features get_text_features"
# https://huggingface.co/docs/transformers/model_doc/clip
#
# ١) `from transformers import CLIPModel, CLIPProcessor` — حمّل كليهما من `CHECKPOINT`،
#    ونادِ `.eval()`، واعمل على دفعات بحجم `IMAGE_BATCH` ليبقى استهلاك الذاكرة ثابتًا.
# ٢) تُعيد `get_image_features` و`get_text_features` كائن خرج في الإصدارات الحالية:
#    والمتّجه في `.pooler_output`، وهو غير مُطبَّع.
# ٣) يحتاج المعالج `padding=True` و`truncation=True` لدفعة تعليقات مختلفة
#    الأطوال.
# ابحث عن: "transformers CLIPModel get_image_features get_text_features"
# https://huggingface.co/docs/transformers/model_doc/clip
# ────────────────────────────────────────────────────────────────────

# TODO: Load CLIP, write embed_images and embed_texts returning unit-length arrays, embed the 200 pairs, and assert the two embedding sets share a dimension.
# مهمة: حمّل CLIP، واكتب `embed_images` و`embed_texts` تُعيدان مصفوفات بطول الوحدة، وضمّن الأزواج المئتين، وافحص أن للمجموعتين البُعد نفسه.

### Task 2.2 — search, both ways, from one matrix

Text → image: pick five captions, and for each return the five most similar images. Display them.

Image → text: pick five images, and for each return the five most similar captions.

**Same matrix, read the other way.** Text-to-image reads a column; image-to-text reads a row.
Nothing was rebuilt, no second model was loaded, and there is no "image search model" and "caption
model" — there is one 200×200 table of numbers and two ways of scanning it. Say that sentence out
loud, because it is the thing that makes a vector database a general-purpose tool rather than a
per-task one.

<div dir="rtl" align="right">

### المهمة ٢٫٢ — البحث في الاتجاهين، من مصفوفة واحدة

من النص إلى الصورة: اختر خمسة تعليقات، وأعِد لكلٍّ أشبه خمس صور. اعرضها.

ومن الصورة إلى النص: اختر خمس صور، وأعِد لكلٍّ أشبه خمسة تعليقات.

**المصفوفة نفسها، مقروءةً في الاتجاه الآخر.** فالبحث من النص إلى الصورة يقرأ عمودًا، ومن الصورة إلى
النص يقرأ صفًّا. ولم يُعَد بناء شيء، ولم يُحمَّل نموذج ثانٍ، وليس ثمّة «نموذج بحث صور» و«نموذج تعليقات»
— بل جدول أعداد واحد ٢٠٠×٢٠٠ وطريقتان لمسحه. قل هذه الجملة بصوت عالٍ، فهي ما يجعل قاعدة بيانات
المتّجهات أداةً عامّة لا أداةً لكل مهمّة.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Write one top_k(scores, k) helper returning indices sorted by descending score —
#    argsort on the negated scores is the shortest way.
# 2) For text -> image take SIMILARITY[:, caption_index]; for image -> text take
#    SIMILARITY[image_index, :]. That is the whole difference.
# 3) Plot the five text queries as five rows of five images, and print the image -> text
#    results rather than plotting them.
# Search: "numpy argsort descending top k retrieval"
# https://numpy.org/doc/stable/reference/generated/numpy.argsort.html
#
# ١) اكتب دالة `top_k(scores, k)` تُعيد الفهارس مرتّبةً تنازليًا بالدرجة — و`argsort` على
#    الدرجات المنفية أقصر طريق.
# ٢) من النص إلى الصورة خذ `SIMILARITY[:, caption_index]`؛ ومن الصورة إلى النص خذ
#    `SIMILARITY[image_index, :]`. وهذا كل الفرق.
# ٣) ارسم الاستعلامات النصية الخمسة خمسة صفوف من خمس صور، واطبع نتائج الصورة إلى النص بدل
#    رسمها.
# ابحث عن: "numpy argsort descending top k retrieval"
# https://numpy.org/doc/stable/reference/generated/numpy.argsort.html
# ────────────────────────────────────────────────────────────────────

# TODO: Write top_k, then show five text -> image queries as a grid and print five image -> text queries, both read from SIMILARITY.
# مهمة: اكتب `top_k`، ثم اعرض خمسة استعلامات من النص إلى الصورة في شبكة واطبع خمسة من الصورة إلى النص، وكلاهما مقروء من `SIMILARITY`.

### Task 2.3 — the metrics, in both directions

Three numbers per direction, on the 200 known-correct pairs.

- **Recall@1** — the fraction of queries whose correct answer came first.
- **Recall@5** — the fraction whose correct answer was in the top five. It is always at least
  Recall@1, which the sanity check verifies; if it is not, the ranking is broken.
- **Mean reciprocal rank** — the average of `1 / rank`. It is the one that notices the difference
  between "second" and "hundredth", which both Recall@1 and Recall@5 record as failures.

Compute all six. **The two directions are not symmetric** — the same matrix, the same pairs, and
different numbers, because the two argmaxes are taken over different axes. Text-to-image asks "of
200 images, which best matches this sentence"; image-to-text asks "of 200 sentences, which best
matches this picture". A picture with several objects has many plausible captions; a caption
usually has one plausible picture. Note the gap and say which direction is harder here.

<div dir="rtl" align="right">

### المهمة ٢٫٣ — المقاييس في الاتجاهين

ثلاثة أعداد لكل اتجاه، على الأزواج المئتين المعروفة الصحّة.

- **الاستدعاء عند ١** — نسبة الاستعلامات التي جاء جوابها الصحيح أولًا.
- **الاستدعاء عند ٥** — نسبة التي كان جوابها الصحيح في الخمسة الأوائل. وهو دائمًا لا يقلّ عن
  الاستدعاء عند ١، ويتحقّق فحص السلامة من ذلك؛ فإن قلّ فالترتيب معطوب.
- **متوسّط الرتبة المتبادلة** — متوسّط `1 / rank`. وهو الذي يلحظ الفرق بين «الثاني» و«المئة»، وكلاهما
  إخفاق عند المقياسين السابقين.

احسب الستّة. **والاتجاهان غير متناظرين** — المصفوفة نفسها والأزواج نفسها وأعداد مختلفة، لأن القيمتين
العظميين تُؤخذان على محورين مختلفين. فالبحث من النص إلى الصورة يسأل «أيّ صورة من مئتين تُطابق هذه
الجملة»؛ ومن الصورة إلى النص يسأل «أيّ جملة من مئتين تُطابق هذه الصورة». وللصورة ذات الأجسام المتعدّدة
تعليقات محتملة كثيرة؛ وللتعليق صورة محتملة واحدة غالبًا. لاحظ الفجوة وقل أيّ الاتجاهين أصعب هنا.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) The rank of the correct answer for query i is where i lands in the sorted order of
#    that query's scores — argsort twice gives you every rank at once.
# 2) Recall@k is (ranks < k).mean() with zero-based ranks; MRR is (1 / (ranks + 1)).mean().
# 3) Do it once as a function of a score matrix, then call it on SIMILARITY and on
#    SIMILARITY.T — the transpose is the other direction, and nothing else changes.
# Search: "recall at k mean reciprocal rank retrieval evaluation"
# https://numpy.org/doc/stable/reference/generated/numpy.argsort.html
#
# ١) رتبة الجواب الصحيح للاستعلام `i` هي موضع `i` في ترتيب درجات ذلك الاستعلام — و`argsort`
#    مرّتين يعطيك كل الرتب دفعةً واحدة.
# ٢) الاستدعاء عند k هو `(ranks < k).mean()`؛ ومتوسّط الرتبة `(1 / (ranks + 1)).mean()`.
# ٣) افعلها مرّةً دالّةً لمصفوفة درجات، ثم نادِها على `SIMILARITY` وعلى `SIMILARITY.T` —
#    فالمنقولة هي الاتجاه الآخر، ولا شيء غير ذلك يتغيّر.
# ابحث عن: "recall at k mean reciprocal rank retrieval evaluation"
# https://numpy.org/doc/stable/reference/generated/numpy.argsort.html
# ────────────────────────────────────────────────────────────────────

# TODO: Write retrieval_metrics(scores) returning R@1, R@5 and MRR from the diagonal being the correct answer, then report it for both directions in one table.
# مهمة: اكتب `retrieval_metrics(scores)` تُعيد الاستدعاء عند ١ وعند ٥ ومتوسّط الرتبة المتبادلة من كون القطر هو الجواب الصحيح، ثم أبلغ عنها للاتجاهين في جدول واحد.

### Task 2.4 — zero-shot classification, and the three-way row

Classify all 400 images of `small_image_5class` with **no training images at all**. Embed the five
class names as `"a photo of a {class}"`, embed the images, take the argmax per image. That is the
whole method.

Then put three numbers in one row, on the same 400 images:

| method | labels used | number |
|---|---|---|
| CLIP zero-shot | **0** | today |
| DINOv2 frozen + linear probe | 320 | Thursday |
| ResNet-18 fine-tuned | 320 | W4D5 |

**Read the result carefully, because it is not the one the plan expected.** CLIP wins here, with
zero labels. That is a real number and it is also a trap: the five classes are `bus, cat, dog,
pizza, zebra` — coarse, visually distinct, and exactly the kind of everyday category that appears
in web alt text a hundred million times. CLIP is being asked the question it was built for.

So the sentence to write is not "zero-shot beats supervised". It is: **on categories the internet
describes constantly, zero-shot costs nothing and wins; on your company's five defect types, it has
never seen the words, and Thursday's probe is the method.** The number that matters in a report is
never the accuracy alone — it is the accuracy plus what the classes were.

<div dir="rtl" align="right">

### المهمة ٢٫٤ — التصنيف بلا تدريب، والصفّ الثلاثي

صنّف صور `small_image_5class` الأربعمئة كلها **بلا صور تدريب إطلاقًا**. ضمّن أسماء الفئات الخمس بصيغة
`"a photo of a {class}"`، وضمّن الصور، وخذ أكبر قيمة لكل صورة. وهذه هي الطريقة كلها.

ثم ضع ثلاثة أعداد في صفٍّ واحد، على الصور الأربعمئة نفسها:

| الطريقة | التسميات المستعملة | الرقم |
|---|---|---|
| CLIP بلا تدريب | **٠** | اليوم |
| DINOv2 مجمّد + فحص خطّي | ٣٢٠ | الخميس |
| ResNet-18 مضبوط دقيقًا | ٣٢٠ | الأسبوع الرابع |

**واقرأ النتيجة بتمعّن، فليست هي التي توقّعتها الخطّة.** فـCLIP يفوز هنا، بصفر تسمية. وهذا رقم حقيقي
وهو أيضًا فخّ: فالفئات الخمس `bus, cat, dog, pizza, zebra` — خشنة، ومتمايزة بصريًا، وهي بالضبط نوع
الفئات اليومية التي تظهر في نصوص الشبكة البديلة مئة مليون مرّة. فـCLIP يُسأل السؤال الذي بُني له.

فالجملة التي تكتبها ليست «ما بلا تدريب يهزم المُشرَف». بل: **على فئات تصفها الشبكة باستمرار، لا يكلّف
ما بلا تدريب شيئًا ويفوز؛ وعلى أنواع العيوب الخمسة في شركتك لم يرَ تلك الكلمات قط، ويكون فحص الخميس
هو الطريقة.** والرقم الذي يهمّ في تقرير ليس الدقّة وحدها أبدًا — بل الدقّة مع بيان ما كانت الفئات.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Embed the five prompts with embed_texts and the 400 images with embed_images, then
#    one dot product gives a (400, 5) score matrix.
# 2) The prediction is CLASSES[argmax] per row — compare against CLASS_LABELS directly.
# 3) D4_PROBE and W4D5_FINETUNE were loaded in the setup cell from Thursday's artefact;
#    put all three in one small DataFrame with a labels-used column.
# Search: "clip zero-shot classification a photo of a prompt"
# https://huggingface.co/docs/transformers/model_doc/clip
#
# ١) ضمّن الموجّهات الخمسة بـ`embed_texts` والصور الأربعمئة بـ`embed_images`، فيُعطي جداء
#    قياسي واحد مصفوفة درجات `(400, 5)`.
# ٢) التنبّؤ هو `CLASSES[argmax]` لكل صف — قارنه بـ`CLASS_LABELS` مباشرةً.
# ٣) حُمّل `D4_PROBE` و`W4D5_FINETUNE` في خلية الإعداد من أثر الخميس؛ ضع الثلاثة في
#    `DataFrame` صغير بعمود لعدد التسميات المستعملة.
# ابحث عن: "clip zero-shot classification a photo of a prompt"
# https://huggingface.co/docs/transformers/model_doc/clip
# ────────────────────────────────────────────────────────────────────

# TODO: Embed the class prompts and all 400 images, classify by argmax, and build the three-way comparison table with the number of labels each method used.
# مهمة: ضمّن موجّهات الفئات والصور الأربعمئة، وصنّف بأكبر قيمة، وابنِ جدول المقارنة الثلاثي مع عدد التسميات التي استعملتها كل طريقة.

### Task 2.5 — prompt sensitivity, measured

Re-run the zero-shot classification with three wordings of the same question: the bare class name,
`"a photo of a {c}"`, and `"a close-up photo of a {c}"`. Add a fourth: the same sentence in Arabic.

Nothing about the images changes. Nothing about the model changes. Only the sentence changes, and
the accuracy moves.

**Here it moves by less than a point**, and that is worth saying plainly rather than dressing up:
this task is saturated, and a saturated benchmark cannot show you prompt sensitivity. The result
still carries the lesson, because the ordering is stable and the Arabic prompt is the worst of the
four — the same asymmetry task 2.6 finds in retrieval.

The rule that survives: **a zero-shot number is a property of a model *and a sentence*.** Anyone
quoting one without the template has left out half the method, and on a harder class set the half
they left out is worth several points, not a fraction of one.

<div dir="rtl" align="right">

### المهمة ٢٫٥ — حساسية الموجّه، مقيسةً

أعِد تشغيل التصنيف بلا تدريب بثلاث صياغات للسؤال نفسه: اسم الفئة مجرّدًا، و`"a photo of a {c}"`، و
`"a close-up photo of a {c}"`. وأضف رابعة: الجملة نفسها بالعربية.

لا شيء يتغيّر في الصور. ولا شيء يتغيّر في النموذج. الجملة وحدها تتغيّر، وتتحرّك الدقّة.

**وهي تتحرّك هنا بأقلّ من نقطة**، ويستحقّ هذا أن يُقال بصراحة لا أن يُزيَّن: فهذه المهمّة مُشبَعة،
والمقياس المُشبَع لا يُظهر لك حساسية الموجّه. ومع ذلك تحمل النتيجة الدرس، لأن الترتيب ثابت والموجّه
العربي أسوأ الأربعة — وهو التفاوت نفسه الذي تجده المهمة ٢٫٦ في الاسترجاع.

والقاعدة التي تبقى: **الرقم بلا تدريب خاصّية لنموذج **ولجملة**.** فمن يذكره بلا القالب قد أسقط نصف
الطريقة، وعلى مجموعة فئات أصعب يساوي النصف الذي أسقطه عدّة نقاط لا كسرًا من نقطة.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Your zero_shot(template) from the previous task already does the work — call it in a
#    loop over the templates and collect the accuracies.
# 2) Report the spread as max minus min in percentage points, and count how many distinct
#    accuracies came out.
# 3) Sort the table so the ordering is visible, and mark which template you would have
#    reported if you had only tried one.
# Search: "clip prompt engineering template zero-shot accuracy"
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html
#
# ١) دالّتك `zero_shot(template)` من المهمة السابقة تؤدّي العمل أصلًا — نادِها في حلقة على
#    القوالب واجمع الدقّات.
# ٢) أبلغ عن المدى فرقًا بين الأكبر والأصغر بنقاط مئوية، وعُدّ كم دقّة متمايزة
#    خرجت.
# ٣) رتّب الجدول ليظهر الترتيب، وأشِر إلى القالب الذي كنت ستُبلّغ عنه لو جرّبت
#    واحدًا فقط.
# ابحث عن: "clip prompt engineering template zero-shot accuracy"
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html
# ────────────────────────────────────────────────────────────────────

# TODO: Score every template in TEMPLATES, tabulate the accuracies, and report the spread and how many distinct values came out.
# مهمة: قيّم كل قالب في `TEMPLATES`، وجدوِل الدقّات، وأبلغ عن المدى وعدد القيم المتمايزة.

### Task 2.6 — find the failures and name the causes

**This is the graded task.** Every model this week was strong. The deliverable is knowing precisely
where each one stops being strong.

The dataset ships eight captions written to be hard, marked in the `hard_case` column: three
counting captions, two spatial relations, one about text inside the image, and one Arabic caption
with an English twin on a different image. Every one is true of its image.

For each, use the caption as a text query and find the rank of its own image among all 200. Rank 1
means CLIP found it; rank 137 means it did not.

Then write **one sentence per failure naming the cause**. The causes are not mysterious and they
are not bugs:

- **Counting.** Contrastive training on alt text rewards knowing that cars are present. Almost no
  caption on the internet is wrong about *whether* there is a car and almost none is checked for
  *how many*, so "three" carries almost no gradient. CLIP is a bag-of-concepts matcher with weak
  binding between a number and a noun.
- **Spatial relations.** Same argument: "the cat under the table" and "the cat on the table" have
  nearly identical bags of concepts, and the training signal that separates them is rare.
- **Text in images.** CLIP does read some text, which is why one of these may well succeed — and
  which is also the OCR-attack story where a note reading "iPod" taped to an apple makes it predict
  iPod.
- **Arabic.** The training corpus is overwhelmingly English. This is a data property, not a
  capability limit, and it is the one that matters most for anything you deploy in Arabic.

**Some of the eight will succeed**, and that is information too — note which, and say what made
them easier. "Two zebras" is easier than "three cars" because the image contains nothing else, and
saying that is a better answer than a list of failures.

<div dir="rtl" align="right">

### المهمة ٢٫٦ — جِد الإخفاقات وسمِّ الأسباب

**هذه هي المهمّة المُقيَّمة.** كان كل نموذج هذا الأسبوع قويًا. والمطلوب أن تعرف بدقّة أين يكفّ كلٌّ منها
عن القوّة.

تحمل المجموعة ثمانية تعليقات كُتبت لتكون صعبة، موسومةً في عمود `hard_case`: ثلاثة تعليقات عدّ،
وعلاقتان مكانيتان، وواحد عن نصّ داخل الصورة، وتعليق عربي مع توأم إنجليزي على صورة أخرى. وكلها صحيحة
في صورها.

استعمل كل تعليق استعلامًا نصيًا وجِد رتبة صورته بين المئتين. فالرتبة ١ تعني أن CLIP وجدها، والرتبة
١٣٧ تعني أنه لم يجدها.

ثم اكتب **جملةً لكل إخفاق تُسمّي سببه**. والأسباب ليست غامضة وليست أخطاءً برمجية:

- **العدّ.** التدريب التقابلي على النصوص البديلة يكافئ معرفة **وجود** سيارات. فلا يكاد تعليق على
  الشبكة يُخطئ في **هل** ثمّة سيارة، ولا يكاد يُفحص في **كم**، فلا تحمل «ثلاث» تدرّجًا يُذكر. وCLIP
  مُطابِق حقيبة مفاهيم بربطٍ ضعيف بين العدد والاسم.
- **العلاقات المكانية.** الحجّة نفسها: «القطّة تحت الطاولة» و«القطّة على الطاولة» حقيبتا مفاهيم شبه
  متطابقتين، وإشارة التدريب التي تفصلهما نادرة.
- **النصّ داخل الصور.** يقرأ CLIP بعض النصّ، ولهذا قد ينجح أحد هذه — وهي أيضًا قصّة هجوم التعرّف
  الضوئي حيث تجعله ورقةٌ مكتوب عليها «iPod» ملصقةٌ على تفّاحة يتنبّأ بـiPod.
- **العربية.** مُدوّنة التدريب إنجليزية بأغلبية ساحقة. وهذه خاصّية بيانات لا حدّ قدرة، وهي الأهمّ لأي
  شيء تُشغّله بالعربية.

**وبعض الثمانية سينجح**، وهذا معلومة أيضًا — دوّن أيّها، وقل ما الذي سهّلها. فـ«حماران وحشيّان» أسهل
من «ثلاث سيارات» لأن الصورة لا تحوي غيرهما، وقول ذلك جواب أفضل من قائمة إخفاقات.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) CAPTIONS[CAPTIONS.hard_case.notna()] gives you the eight rows and their kinds.
# 2) The rank of image i for caption i is its position in argsort(-SIMILARITY[:, i]) —
#    add 1 so the best possible rank reads as 1 rather than 0.
# 3) Also record what CLIP returned instead, so the diagnosis has evidence: the top-1
#    image's own caption is usually enough to name the cause.
# Search: "clip counting spatial reasoning failure bag of words"
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.itertuples.html
#
# ١) يعطيك `CAPTIONS[CAPTIONS.hard_case.notna()]` الصفوف الثمانية وأنواعها.
# ٢) رتبة الصورة `i` للتعليق `i` هي موضعها في `argsort(-SIMILARITY[:, i])` — أضف واحدًا
#    لتُقرأ أفضل رتبة ١ لا ٠.
# ٣) وسجّل أيضًا ما أعاده CLIP بدلًا منها ليكون للتشخيص دليل: فتعليق الصورة الأولى يكفي
#    عادةً لتسمية السبب.
# ابحث عن: "clip counting spatial reasoning failure bag of words"
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.itertuples.html
# ────────────────────────────────────────────────────────────────────

# TODO: For every hard_case caption, find the rank of its own image among all 200, record what came first instead, and display the eight with their ranks.
# مهمة: لكل تعليق في `hard_case`، جِد رتبة صورته بين المئتين، وسجّل ما جاء أولًا بدلًا منها، واعرض الثمانية برتبها.

**Your diagnoses.** One sentence per failure, naming the cause. Then one more sentence on the hard
cases that *succeeded* — what made them easier than the ones that did not?

<div dir="rtl" align="right">

**تشخيصاتك.** جملة لكل إخفاق تُسمّي سببه. ثم جملة أخرى عن الحالات الصعبة التي **نجحت** — ما الذي
جعلها أسهل من التي لم تنجح؟

</div>

## Section 3 — Stretch: the search tool, and the number that motivates week 7  (≈30 min)

Build the thing. A function that takes a free-text query and returns the top five images with their
scores, against a corpus embedded once and cached. Twenty lines, and it is a working image search
engine over 200 photographs.

Then time it, honestly. Time one query against a brute-force scan of all 200 vectors, and
extrapolate to a million: brute force is linear, so the estimate is `time × (1,000,000 / 200)`.
Write the number down. That number is why W7D2 exists.

Two things to say about the estimate rather than around it. It is optimistic — a real corpus does
not fit in one cache-friendly NumPy array, and the constant gets worse, not better, at scale. And
the *embedding* cost is not in it at all: the corpus is embedded once, and that one-off is minutes
per million images on a CPU, which is a different budget from the per-query one and is the reason
"embed once, query many" is the shape of every retrieval system you will build.

**The handover.** Next week the flat scan becomes a vector index (W7D2), a language model goes in
front of it (W7D3), and the pair becomes retrieval-augmented generation. Everything you built today
survives that: the embedding, the cosine, the top-k, and the habit of quoting the corpus size next
to the recall.

<div dir="rtl" align="right">

## القسم الثالث — التوسّع: أداة البحث، والرقم الذي يُبرّر الأسبوع السابع (نحو ٣٠ دقيقة)

ابنِ الشيء نفسه. دالّة تأخذ استعلامًا نصيًا حرًّا وتُعيد أفضل خمس صور بدرجاتها، على مُدوّنة مُضمَّنة
مرّةً ومُخزَّنة. عشرون سطرًا، وهي محرّك بحث صور عامل على مئتَي صورة.

ثم قِس زمنها بصدق. قِس زمن استعلام واحد على مسحٍ شامل للمتّجهات المئتين، واستقرِ إلى مليون: فالمسح
الشامل خطّي، والتقدير `الزمن × (1,000,000 / 200)`. دوّن الرقم. فذلك الرقم هو سبب وجود الأسبوع السابع
اليوم الثاني.

وأمران يُقالان عن التقدير لا حوله. إنه متفائل — فالمُدوّنة الحقيقية لا تسع مصفوفة NumPy واحدة صديقةً
للذاكرة المخبّأة، والثابت يسوء لا يتحسّن مع الحجم. وكلفة **التضمين** ليست فيه أصلًا: فالمُدوّنة تُضمَّن
مرّةً، وتلك المرّة دقائق لكل مليون صورة على المعالج، وهي ميزانية غير ميزانية الاستعلام، وهي سبب كون
«ضمّن مرّة واستعلم كثيرًا» شكلَ كل نظام استرجاع ستبنيه.

**التسليم.** يصير المسح المسطّح الأسبوع القادم فهرس متّجهات (الأسبوع السابع اليوم الثاني)، ويقف أمامه
نموذج لغوي (اليوم الثالث)، ويصير الاثنان توليدًا معزّزًا بالاسترجاع. وكل ما بنيته اليوم ينجو من ذلك:
التضمين، وجيب التمام، وأفضل k، وعادة ذكر حجم المُدوّنة بجوار الاستدعاء.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) The corpus is IMAGE_EMBEDDINGS, already unit-length — a search is embed_texts on one
#    string, one dot product, and top_k.
# 2) Time the scan only, not the text embedding: they are different budgets and mixing
#    them makes the extrapolation meaningless.
# 3) Repeat the scan a few hundred times to get past timer resolution, then scale by
#    1_000_000 / len(corpus).
# Search: "brute force vector search cost linear scan extrapolate"
# https://docs.python.org/3/library/time.html#time.perf_counter
#
# ١) المُدوّنة هي `IMAGE_EMBEDDINGS` وهي بطول الوحدة أصلًا — فالبحث `embed_texts` على نصٍّ
#    واحد، وجداء قياسي، و`top_k`.
# ٢) قِس زمن المسح وحده لا تضمين النصّ: فهما ميزانيتان مختلفتان وخلطهما يُبطل
#    الاستقراء.
# ٣) كرّر المسح بضع مئات من المرّات لتتجاوز دقّة المؤقّت، ثم قِس بـ
#    `1_000_000 / len(corpus)`.
# ابحث عن: "brute force vector search cost linear scan extrapolate"
# https://docs.python.org/3/library/time.html#time.perf_counter
# ────────────────────────────────────────────────────────────────────

# TODO: Write search(query, k) over the cached corpus, run three queries, then time the scan and extrapolate the cost of one query against a million vectors.
# مهمة: اكتب `search(query, k)` على المُدوّنة المُخزَّنة، وشغّل ثلاثة استعلامات، ثم قِس زمن المسح واستقرِ كلفة استعلام واحد على مليون متّجه.

## Save your artefact

`clip_similarity.parquet` — every query in the lab: the query text, the direction, the top-k ids
and their scores.

`retrieval_report.md` — the week's closing document, and the one to keep: the six warm-up numbers,
the metrics table in both directions, the three-way zero-shot comparison **including the row where
a supervised method loses**, the prompt sweep, and the diagnosed failures.

Write the report so that someone who was not here can read it without the notebook. That is the
actual deliverable of a retrieval evaluation, and it is the format every model report you write
after this course will take.

<div dir="rtl" align="right">

## احفظ أثرك

`clip_similarity.parquet` — كل استعلام في المعمل: نصّ الاستعلام، والاتجاه، ومعرّفات أفضل k ودرجاتها.

`retrieval_report.md` — وثيقة ختام الأسبوع، وهي التي تُحتفظ: الأعداد الستّة من الإحماء، وجدول
المقاييس في الاتجاهين، ومقارنة بلا تدريب الثلاثية **بما فيها الصفّ الذي تخسر فيه طريقة مُشرَفة**،
ومسح الموجّهات، والإخفاقات المُشخَّصة.

اكتب التقرير بحيث يقرأه من لم يحضر بلا الدفتر. فذلك هو المُنتَج الحقيقي لتقييم استرجاع، وهو الشكل
الذي سيأخذه كل تقرير نموذج تكتبه بعد هذه الدورة.

</div>

In [ ]:
rows = []
for query in QUERY_ROWS:
    for direction, scores, names in [
        ("text → image", SIMILARITY[:, query], CAPTIONS.file.tolist()),
        ("image → text", SIMILARITY[query, :], CAPTIONS.caption.tolist()),
    ]:
        chosen = top_k(scores, 5)
        rows.append({"query": CAPTIONS.caption[query] if direction == "text → image"
                     else CAPTIONS.file[query],
                     "direction": direction,
                     "top_k_ids": ", ".join(str(int(i)) for i in chosen),
                     "top_k_scores": ", ".join(f"{scores[i]:.3f}" for i in chosen),
                     "correct_rank": int(np.flatnonzero(np.argsort(-scores) == query)[0]) + 1})

SIMILARITY_PATH = ARTEFACT_DIR / "clip_similarity.parquet"
pd.DataFrame(rows).to_parquet(SIMILARITY_PATH, index=False)

report_text = f"""# W6D5 — Cross-modal retrieval report

Model: `{CHECKPOINT}` · corpus: `image_captions_small`, {len(CAPTIONS)} image–caption pairs ·
zero-shot set: `small_image_5class`, {len(CLASS_PATHS)} images, classes {CLASSES}.

## 1. The warm-up matrix

```
{pd.DataFrame(WARMUP_MATRIX.round(3), index=IMAGE_NAMES, columns=TEXT_NAMES).to_string()}
```

The vague caption `t3` scores {WARMUP_MATRIX[0, 2]:.3f} against both images — accurate about both,
useful for neither.

## 2. Retrieval metrics, {len(CAPTIONS)} known-correct pairs

```
{METRICS.round(4).to_string(index=False)}
```

The two directions are not symmetric: {HARDER} is the harder one here. Random guessing on this
corpus would score R@1 = {1 / len(CAPTIONS):.3f}.

## 3. Zero-shot against supervised, same {len(CLASS_PATHS)} images

```
{COMPARISON.to_string(index=False)}
```

CLIP wins with zero labels — on five coarse everyday categories, which is the question it was built
for. On class names the web does not describe, a frozen-feature probe is the method.

## 4. Prompt sensitivity

```
{PROMPTS.to_string(index=False)}
```

Spread {PROMPT_SPREAD * 100:.2f} points across {DISTINCT_PROMPT_SCORES} distinct values. The task
is saturated, so the spread is small; a zero-shot number is still a property of a model *and a
sentence*.

## 5. Diagnosed failures

```
{DIAGNOSIS[["kind", "rank_of_true_image", "found", "caption"]].to_string(index=False)}
```

{len(FAILURES)} of {len(DIAGNOSIS)} hard cases ranked outside the top {FAILURE_RANK} of
{len(CAPTIONS)}. The Arabic caption ranks {arabic.rank_of_true_image} against its English twin's
{twin.rank_of_true_image} — a gap of {ARABIC_GAP} places on equivalent sentences, caused by an
English-dominated training corpus rather than by anything about the sentence.

## 6. Cost

A brute-force scan of {len(IMAGE_EMBEDDINGS)} vectors takes {SCAN_SECONDS * 1e6:.0f} µs;
extrapolated linearly, one query against 1,000,000 vectors is {MILLION_SECONDS * 1000:.0f} ms.
That is the motivation for a vector index.
"""

REPORT_PATH = ARTEFACT_DIR / "retrieval_report.md"
REPORT_PATH.write_text(report_text, encoding="utf-8")

print(report_text[:1200])
print(f"\n...\n\nwrote {REPORT_PATH.name} ({len(report_text.splitlines())} lines) and "
      f"{SIMILARITY_PATH.name} ({len(rows)} rows)")

## Sanity check

<div dir="rtl" align="right">

## فحص سلامة

</div>

In [ ]:
check(np.allclose(WARMUP_MATRIX[0], [0.991, 0.348, 0.781], atol=0.005) and VAGUE_IDENTICAL,
      f"the warm-up matrix's first row must be [0.991, 0.348, 0.781] to 2 decimals and the vague "
      f"caption must score identically against both images — got "
      f"{WARMUP_MATRIX[0].round(3).tolist()}, identical: {VAGUE_IDENTICAL}",
      f"يجب أن يكون الصف الأول من مصفوفة الإحماء ‎[0.991, 0.348, 0.781]‎ إلى منزلتين وأن يسجّل "
      f"التعليق الغامض القيمة نفسها مقابل الصورتين — والناتج {WARMUP_MATRIX[0].round(3).tolist()}، "
      f"والتطابق: {VAGUE_IDENTICAL}")

check(SAME_DIMENSION and UNIT_NORM,
      f"image and text embeddings must share a dimension and both be unit-norm to 1e-5 — got "
      f"{IMAGE_EMBEDDINGS.shape[1]} and {TEXT_EMBEDDINGS.shape[1]}, unit norm: {UNIT_NORM}. The "
      f"shared space is the premise of the whole lab",
      f"يجب أن يشترك تضمينا الصور والنصّ في البُعد وأن يكونا بطول الوحدة إلى ‎1e-5‎ — والناتج "
      f"{IMAGE_EMBEDDINGS.shape[1]} و{TEXT_EMBEDDINGS.shape[1]}، وطول الوحدة: {UNIT_NORM}. "
      f"والفضاء المشترك هو مُقدَّمة المعمل كله")

check(bool(SIMILARITY.min() >= -1.0 - 1e-6 and SIMILARITY.max() <= 1.0 + 1e-6),
      f"every similarity must lie in [-1, 1] — got [{SIMILARITY.min():.4f}, "
      f"{SIMILARITY.max():.4f}]. Outside that range means the vectors were not normalised",
      f"يجب أن يقع كل تشابه في المجال ‎[−1, 1]‎ — والناتج [{SIMILARITY.min():.4f}, "
      f"{SIMILARITY.max():.4f}]. والخروج عن المجال يعني أن المتّجهات لم تُطبَّع")

check(RECALL_ORDERED and DIRECTIONS_DIFFER,
      f"R@5 must be at least R@1 in both directions, and the two directions must give different "
      f"numbers — got\n{METRICS.round(4).to_string(index=False)}",
      f"يجب ألّا يقلّ الاستدعاء عند ٥ عن الاستدعاء عند ١ في الاتجاهين، وأن يُعطي الاتجاهان أعدادًا "
      f"مختلفة — والناتج\n{METRICS.round(4).to_string(index=False)}")

check(DISTINCT_PROMPT_SCORES >= 2,
      f"the prompt templates must produce at least two distinct accuracies — got "
      f"{PROMPTS.accuracy.tolist()} for {PROMPTS.template.tolist()}. If they are all identical, "
      f"every template was embedded the same way and the sweep did not run",
      f"يجب أن تُنتج قوالب الموجّه دقّتين متمايزتين على الأقلّ — والناتج {PROMPTS.accuracy.tolist()} "
      f"للقوالب {PROMPTS.template.tolist()}. فإن تطابقت كلها فقد ضُمِّن كل قالب بالطريقة نفسها ولم "
      f"يجرِ المسح")

check(len(FAILURES) >= 3 and "Diagnosed failures" in report_text,
      f"at least three hard cases must rank outside the top {FAILURE_RANK}, and the diagnosis must "
      f"be written into retrieval_report.md — got {len(FAILURES)} failures of {len(DIAGNOSIS)}. "
      f"A student who can only demonstrate a model working has not evaluated it",
      f"يجب أن تقع ثلاث حالات صعبة على الأقلّ خارج أفضل {FAILURE_RANK}، وأن يُكتب التشخيص في "
      f"`retrieval_report.md` — والناتج {len(FAILURES)} من {len(DIAGNOSIS)}. فمن لا يستطيع إلا "
      f"إظهار نموذجٍ يعمل لم يُقيّمه")

check(ARABIC_GAP > 0,
      f"the Arabic caption must rank worse than its English twin — got "
      f"{arabic.rank_of_true_image} against {twin.rank_of_true_image}. That gap is the dataset's "
      f"recorded finding, and it is a property of CLIP's training corpus, not of the file",
      f"يجب أن تكون رتبة التعليق العربي أسوأ من رتبة توأمه الإنجليزي — والناتج "
      f"{arabic.rank_of_true_image} مقابل {twin.rank_of_true_image}. وتلك الفجوة هي نتيجة المجموعة "
      f"المُسجَّلة، وهي خاصّية لمُدوّنة تدريب CLIP لا للملف")

check(len(COMPARISON) == 3 and COMPARISON.labels_used.min() == 0
      and set(COMPARISON.accuracy.round(4)) >= {round(ZERO_SHOT, 4), round(D4_PROBE, 4)},
      f"the zero-shot row must record all three comparison numbers including the ones where a "
      f"supervised method loses — got\n{COMPARISON.to_string(index=False)}",
      f"يجب أن يُسجّل صفّ ما بلا تدريب أرقام المقارنة الثلاثة بما فيها التي تخسر فيها طريقة مُشرَفة — "
      f"والناتج\n{COMPARISON.to_string(index=False)}")

report()

## What's next

**Week 7 — retrieval, RAG and recommenders.** The flat scan you just timed becomes a vector index
on W7D2, a language model goes in front of it on W7D3, and the pair is retrieval-augmented
generation: the model answers from documents it retrieved rather than from what it memorised.

Everything from today survives into it unchanged — embed once, cosine, top-k, and the discipline of
quoting the corpus size next to the recall and the class list next to the accuracy.

**Capstone M2 is due today.** If your project has images and few labels, the week's answer is
Thursday's: frozen features and a probe first. If it needs search rather than classification, it is
today's, and `retrieval_report.md` is the shape your evaluation should take.

<div dir="rtl" align="right">

## ما التالي

**الأسبوع السابع — الاسترجاع والتوليد المعزّز والتوصية.** يصير المسح المسطّح الذي قِست زمنه فهرس
متّجهات في اليوم الثاني، ويقف أمامه نموذج لغوي في اليوم الثالث، ويكون الاثنان توليدًا معزّزًا
بالاسترجاع: فيُجيب النموذج من وثائق استرجعها لا ممّا حفظه.

وكل ما في اليوم ينجو إليه بلا تغيير — ضمّن مرّة، وجيب التمام، وأفضل k، وانضباط ذكر حجم المُدوّنة بجوار
الاستدعاء وقائمة الفئات بجوار الدقّة.

**ويُسلَّم إنجاز مشروع التخرّج الثاني اليوم.** فإن كان في مشروعك صور وتسميات قليلة فجواب الأسبوع هو
جواب الخميس: تمثيلات مجمّدة وفحص أولًا. وإن كان يحتاج بحثًا لا تصنيفًا فجوابه جواب اليوم،
و`retrieval_report.md` هو الشكل الذي ينبغي أن يأخذه تقييمك.

</div>